In [0]:
%sql
CREATE DATABASE IF NOT EXISTS silver;

In [0]:
from pyspark.sql.functions import col, upper, row_number
from pyspark.sql.window import Window

consumer_mapping = {
    "customer_id": "id_consumidor",
    "customer_zip_code_prefix": "prefixo_cep",
    #OBS: customer_name não existe no dataset, porém como estava na atividade inclui no dicionário de mapeamento. Quando o rename executar, nada acontecerá com esta coluna
    "customer_name": "nome_consumidor",
    "customer_city": "cidade",
    "customer_state": "estado"
}

df = spark.table("bronze.tb_customers")
for old_col, new_col in consumer_mapping.items():
    df = df.withColumnRenamed(old_col, new_col)

window = Window.partitionBy("id_consumidor").orderBy(col("timestamp_ingestion").desc())

#Coluna customer_unique_id não consta no mapeamento, então foi removida
df = df.withColumn("row_num", row_number().over(window)) \
       .filter(col("row_num") == 1) \
       .drop("row_num", "timestamp_ingestion", "customer_unique_id")

#Os estados já estão em Uppercase no dataset, mas para garantir a padronização faço o upper() de qualquer forma
df = df.withColumn("cidade", upper(col("cidade"))) \
       .withColumn("estado", upper(col("estado")))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.dim_consumidores")

In [0]:
from pyspark.sql.functions import when, datediff, to_timestamp

orders_mapping = {
    "order_id": "id_pedido",
    "customer_id": "id_consumidor",
    "order_status": "status",
    "order_purchase_timestamp": "data_hora_pedido",
    "order_approved_at": "data_aprovacao_pedido",
    "order_delivered_carrier_date": "data_entrega_transportadora",
    "order_delivered_customer_date": "data_entrega_consumidor",
    "order_estimated_delivery_date": "data_entrega_estimada"
}

df = spark.table("bronze.tb_orders")

for old_col, new_col in orders_mapping.items():
    df = df.withColumnRenamed(old_col, new_col)

df = df.withColumn("status",
    when(col("status") == "delivered", "entregue")
    .when(col("status") == "canceled", "cancelado")
    .when(col("status") == "shipped", "enviado")
    .when(col("status") == "processing", "processando")
    .when(col("status") == "invoiced", "faturado")
    .when(col("status") == "unavailable", "indisponível")
    .when(col("status") == "created", "criado")
    .when(col("status") == "approved", "aprovado")
    .otherwise("desconhecido")
)

df = df.withColumn("tempo_entrega_dias", datediff(col("data_entrega_consumidor"), col("data_hora_pedido")))

df = df.withColumn("tempo_entrega_estimado_dias", datediff(col("data_entrega_estimada"), col("data_hora_pedido")))

df = df.withColumn("diferenca_entrega_dias", datediff(col("data_entrega_consumidor"), col("data_entrega_estimada")))

df = df.withColumn("entrega_no_prazo",
    when(col("status") != "entregue", "Não Entregue")
    .when(col("diferenca_entrega_dias") <= 0, "Sim")
    .otherwise("Não")
)

df = df.drop("timestamp_ingestion")

#Em caso de o inferSchema não ter identificado corretamente as colunas de data
colunas_data = [
    "data_hora_pedido",
    "data_aprovacao_pedido",
    "data_entrega_transportadora",
    "data_entrega_consumidor",
    "data_entrega_estimada"
]

for c in colunas_data:
    df = df.withColumn(c, to_timestamp(col(c)))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.fat_pedidos")

In [0]:
order_items_mapping = {
  "order_id": "id_pedido",
  "order_item_id": "id_item",
  "product_id": "id_produto",
  "seller_id": "id_vendedor",
  "price": "preco_BRL",
  "freight_value": "preco_frete"
}

df = spark.table("bronze.tb_order_items")

#Coluna Shipping_limit_date não consta no mapeamento, então foi removida
df = df.drop("shipping_limit_date", "timestamp_ingestion")

for old_col, new_col in order_items_mapping.items():
  df = df.withColumnRenamed(old_col, new_col)

number_type_mapping = [
  "preco_BRL",
  "preco_frete"
]

for i in number_type_mapping:
  df = df.withColumn(i, col(i).cast("decimal(10,2)"))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.fat_itens_pedidos")

In [0]:
order_payments_mapping = {
  "order_id": "id_pedido",
  "payment_sequential": "sequencial_pagamento",
  "payment_type": "tipo_pagamento",
  "payment_installments": "parcelas_pagamento",
  "payment_value": "valor_pagamento"
}

df = spark.table("bronze.tb_order_payments")

for old_col, new_col in order_payments_mapping.items():
  df = df.withColumnRenamed(old_col, new_col)

df = df.withColumn("tipo_pagamento",
    when(col("tipo_pagamento") == "credit_card", "Cartão de Crédito")
    .when(col("tipo_pagamento") == "boleto", "Boleto")
    .when(col("tipo_pagamento") == "voucher", "Voucher")
    .when(col("tipo_pagamento") == "debit_card", "Cartão de Débito")
    .when(col("tipo_pagamento") == "not_defined", "Não Definido")
    .otherwise("desconhecido")
)

df = df.drop("timestamp_ingestion")

number_type_mapping = [
    "valor_pagamento"
]

for i in number_type_mapping:
  df = df.withColumn(i, col(i).cast("decimal(10,2)"))

df.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("silver.fat_pagamentos_pedidos")